# 3.1 怎麼彙整多個位置？

# 第 3 章：Attention，自己決定看哪裡

前置：向量、矩陣乘法和 softmax。先用單 head；每個小節都畫得出矩陣。完整模型仍留到第4章。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：問路譬喻：想找什麼、誰知道、拿回什麼**

Q/K 決定讀哪裡；V 是讀回的內容。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/qkv.svg")))

**先想一想：**[0.9,0.1] 會讓結果靠近哪一個向量？

先用已知權重做平均：想讀第一個位置，就把它的權重提高。attention 先是這件簡單的事，後面才讓內容決定權重。

**把直覺寫成數學：**$o=\sum_j a_j v_j$；$a_j\ge0$ 且總和1，o 是加權平均。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
values = torch.tensor([[2.0, 0.0], [0.0, 4.0]])
weights = torch.tensor([0.9, 0.1])
print(weights @ values)
assert torch.allclose(weights @ values, torch.tensor([1.8, 0.4]))

**如何讀結果：**權重控制混合比例，不需要先把所有向量平均再挑。

**只改一件事：**只交換兩個權重。
